# llm_judge — Q5 (les modeles devinent l'auteur)
Runtime -> GPU. Le juge par defaut est un modele open-weight local.

In [1]:
# 1) Dependances
!pip -q install transformers accelerate scikit-learn pandas numpy torch
import torch; print('GPU:',torch.cuda.is_available())

GPU: True


## 2) (Optionnel) Login HuggingFace

In [2]:
from huggingface_hub import notebook_login
notebook_login()

## 3) Charge dataset_clean.csv

In [3]:
from google.colab import files
up = files.upload()   # -> dataset_clean.csv

Saving dataset_clean.csv to dataset_clean.csv


## 4) Le juge

In [4]:
"""
llm_judge.py
------------------------------------------------------------------
Partie Q5 : "LLM as a judge".

On demande a des modeles-JUGES de deviner qui a ecrit un texte (parmi une
liste d'auteurs candidats : human + chaque modele), et d'expliquer leur choix.
On mesure l'accuracy du juge et on garde les explications (analyse qualitative).

Bonus "self-recognition" : si un juge est aussi un des generateurs, on regarde
s'il reconnait ses PROPRES textes mieux que le hasard.

Juges possibles :
  - "hf"     : modele open-weight local (implemente)
  - "openai" : API OpenAI  (stub : mettez votre cle dans OPENAI_API_KEY)
  - "gemini" : API Gemini  (stub : mettez votre cle dans GOOGLE_API_KEY)

Dependances : pip install torch transformers pandas scikit-learn
              (+ openai / google-genai si vous utilisez les juges payants)
------------------------------------------------------------------
"""

import os
import re
import gc
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# ============================================================
# CONFIG
# ============================================================
DATASET_CSV = "dataset_clean.csv"   # le dataset nettoye
N_SAMPLE    = 60            # nb de textes juges (juger coute cher -> on echantillonne)
MAX_NEW     = 120

# Liste des juges a tester
JUDGES = [
    {"name": "Qwen2.5-1.5B-Instruct", "type": "hf",     "model": "Qwen/Qwen2.5-1.5B-Instruct"},
    # {"name": "gpt-5.4-nano",        "type": "openai", "model": "gpt-5.4-nano"},
    # {"name": "gemini-3.5-flash-lite","type":"gemini", "model": "gemini-3.5-flash-lite"},
]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.float16 if DEVICE == "cuda" else torch.float32


# ============================================================
# PROMPT
# ============================================================
def build_prompt(text, candidates):
    liste = "\n".join(f"- {c}" for c in candidates)
    return (
        "You are given a text. It was written by exactly ONE of these authors:\n"
        f"{liste}\n\n"
        "Guess which author wrote it. Answer EXACTLY in this format:\n"
        "Author: <one name copied from the list>\n"
        "Reason: <one short sentence>\n\n"
        f'TEXT:\n"""{text}"""'
    )

def parse_author(answer, candidates):
    """Recupere le nom d'auteur choisi dans la reponse du juge."""
    m = re.search(r"Author\s*:\s*(.+)", answer, flags=re.I)
    zone = m.group(1) if m else answer
    zone_l = zone.lower()
    # correspondance par nom present dans la reponse
    for c in candidates:
        if c.lower() in zone_l:
            return c
    for c in candidates:                     # repli : n'importe ou dans la reponse
        if c.lower() in answer.lower():
            return c
    return "unknown"

def reason_of(answer):
    m = re.search(r"Reason\s*:\s*(.+)", answer, flags=re.I | re.S)
    return (m.group(1).strip().replace("\n", " ")[:300]) if m else ""


# ============================================================
# GENERATION SELON LE TYPE DE JUGE
# ============================================================
def hf_judge(model_id, prompts):
    tok = AutoTokenizer.from_pretrained(model_id)
    # chargement simple sur un seul device (juge petit -> tient sur le GPU)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=DTYPE).to(DEVICE).eval()
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    answers = []
    for p in prompts:
        msg = [{"role": "user", "content": p}]
        if tok.chat_template:
            text = tok.apply_chat_template(msg, add_generation_prompt=True, tokenize=False)
        else:
            text = p
        enc = tok(text, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                                  pad_token_id=tok.pad_token_id)
        answers.append(tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True))
    del model; gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return answers

def openai_judge(model_id, prompts):
    from openai import OpenAI                       # pip install openai
    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    answers = []
    for p in prompts:
        r = client.chat.completions.create(
            model=model_id, messages=[{"role": "user", "content": p}],
            max_tokens=MAX_NEW, temperature=0)
        answers.append(r.choices[0].message.content)
    return answers

def gemini_judge(model_id, prompts):
    from google import genai                        # pip install google-genai
    client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
    answers = []
    for p in prompts:
        r = client.models.generate_content(model=model_id, contents=p)
        answers.append(r.text)
    return answers

DISPATCH = {"hf": hf_judge, "openai": openai_judge, "gemini": gemini_judge}


# ============================================================
# MAIN
# ============================================================
def main():
    df = pd.read_csv(DATASET_CSV)
    df = df[df["text"].astype(str).str.split().apply(len) >= 20]

    # echantillon stratifie par auteur reel
    sample = (df.groupby("model_name", group_keys=False)
                .apply(lambda g: g.sample(min(len(g), max(1, N_SAMPLE // df['model_name'].nunique())),
                                          random_state=42))
                .reset_index(drop=True))

    candidates = sorted(df["model_name"].unique().tolist())   # human + chaque modele
    prompts = [build_prompt(t, candidates) for t in sample["text"].tolist()]

    for judge in JUDGES:
        print(f"\n================ JUGE : {judge['name']} ({judge['type']}) ================")
        try:
            raw = DISPATCH[judge["type"]](judge["model"], prompts)
        except Exception as e:
            import traceback
            traceback.print_exc()               # montre l'erreur complete
            print(f"  Juge indisponible ({type(e).__name__}: {e}) -> saute")
            continue

        pred = [parse_author(a, candidates) for a in raw]
        true = sample["model_name"].tolist()

        acc = accuracy_score(true, pred)
        f1  = f1_score(true, pred, average="macro")
        print(f"accuracy={acc:.3f}  macro-F1={f1:.3f}  (hasard={1/len(candidates):.3f})")
        print("Confusion (lignes=vrai, cols=predit), ordre =", candidates)
        print(confusion_matrix(true, pred, labels=candidates))

        # self-recognition : le juge reconnait-il ses propres textes ?
        if judge["name"] in candidates:
            own = [i for i, t in enumerate(true) if t == judge["name"]]
            if own:
                rec = np.mean([pred[i] == judge["name"] for i in own])
                print(f"Self-recognition : {rec:.3f} sur {len(own)} de ses propres textes")

        # sauvegarde des reponses + explications
        out = sample[["id", "domain", "model_name"]].copy()
        out["predicted"] = pred
        out["reason"] = [reason_of(a) for a in raw]
        out.to_csv(f"judge_{judge['name']}.csv", index=False)

    print("\nOK -> reponses des juges sauvegardees (judge_*.csv)")


## 5) Lance + telecharge

In [5]:
main()

import glob
from google.colab import files
for f in glob.glob('judge_*.csv'): files.download(f)

/tmp/ipykernel_650/3678999839.py:141: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(len(g), max(1, N_SAMPLE // df['model_name'].nunique())),



================ JUGE : Qwen2.5-1.5B-Instruct (hf) ================


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

accuracy=0.200  macro-F1=0.139  (hasard=0.167)
Confusion (lignes=vrai, cols=predit), ordre = ['Mistral-7B-Instruct-v0.3', 'Phi-4-mini-instruct', 'Qwen3-4B-Instruct-2507', 'SmolLM3-3B', 'gemma-4-E4B-it', 'human']
[[0 0 4 0 2 4]
 [0 0 7 0 2 1]
 [0 0 5 0 0 5]
 [0 0 6 0 0 4]
 [0 1 4 0 2 3]
 [0 4 1 0 0 5]]

OK -> reponses des juges sauvegardees (judge_*.csv)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>